# Week 8 - Wednesday Assignment

**Name:** <your name>  
**Program:** PG Diploma AI-ML and Agentic AI Engineering  
**Date:** 2026-04-16

This notebook covers Sub-steps 1-5 using the provided `hospital_records.csv` + MNIST:
1. Hospital dataset loading + characterization
2. MNIST loading + characterization
3. CNN on MNIST (>=2 conv layers) + first-layer filter visualization
4. Hospital readmission detector + semantic similarity retrieval
5. Two-stage risk pipeline + recommendation at 100,000 cases/day

---

## AI usage log (required by policy)
- Prompt(s) used:
- What AI generated:
- What I changed and why:


In [ ]:
# Core imports and global config
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

HOSPITAL_CSV_PATH = Path("./hospital_records.csv")

print("Notebook initialized")

## 0) Load / validate dataset paths
This cell validates local `hospital_records.csv` and keeps MNIST loading via `torchvision` in Sub-step 2.


In [ ]:
hospital_csv_path = HOSPITAL_CSV_PATH

if not hospital_csv_path.exists():
    raise FileNotFoundError(
        f"Expected file not found: {hospital_csv_path.resolve()}\n"
        "Place hospital_records.csv in the same folder as this notebook."
    )

print("hospital_csv_path:", hospital_csv_path.resolve())

## 1) Sub-step 1: Load and characterize hospital dataset


In [ ]:
hospital_df = pd.read_csv(hospital_csv_path)
print("Shape:", hospital_df.shape)
display(hospital_df.head())

print("\nColumns:", list(hospital_df.columns))
print("\nMissing values per column:")
display(hospital_df.isna().sum().sort_values(ascending=False))

# Standardize common noisy categorical entries
hospital_df["gender_clean"] = hospital_df["gender"].astype(str).str.strip().str.lower().replace(
    {"m": "male", "f": "female", "male": "male", "female": "female"}
)
hospital_df["department_clean"] = hospital_df["department"].astype(str).str.strip().str.lower()

print("\nGender distribution:")
display(hospital_df["gender_clean"].value_counts(dropna=False))
print("\nDepartment distribution:")
display(hospital_df["department_clean"].value_counts(dropna=False).head(15))

# Target distribution for risk modeling
TARGET_COL = "readmitted_30d"
print("\nReadmission class distribution:")
display(hospital_df[TARGET_COL].value_counts(dropna=False))

plt.figure(figsize=(5, 4))
sns.countplot(data=hospital_df, x=TARGET_COL)
plt.title("30-day readmission label distribution")
plt.tight_layout()
plt.show()

### Observations for Sub-step 1 (fill after running)
- Data quality issues found (e.g., mixed date formats, noisy gender values, BMI as string):
- Readmission class distribution issue and why it matters:
- How this affects evaluation choices in Sub-step 5 (precision/recall trade-off):


## 2) Sub-step 2: Load and characterize MNIST
This section uses `torchvision.datasets.MNIST` (recommended baseline) even though a Kaggle MNIST CSV is also downloaded.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_ds = datasets.MNIST(root=DATA_DIR / "mnist_torch", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root=DATA_DIR / "mnist_torch", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

print("Train size:", len(train_ds))
print("Test size:", len(test_ds))

# Distribution across classes
train_targets = train_ds.targets.numpy()
class_counts = pd.Series(train_targets).value_counts().sort_index()
display(class_counts.to_frame("count"))

# Image properties
sample_img, sample_label = train_ds[0]
print("Sample shape:", sample_img.shape, "label:", sample_label)
print("Pixel range:", float(sample_img.min()), "to", float(sample_img.max()))

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    img, lbl = train_ds[i]
    ax.imshow(img.squeeze(0), cmap="gray")
    ax.set_title(str(lbl))
    ax.axis("off")
plt.tight_layout()
plt.show()

### Observations for Sub-step 2 (fill after running)
- Digit distribution:
- Image dimensions:
- Pixel range and normalization/preprocessing decisions:


## 3) Sub-step 3: Build and train CNN on MNIST
Architecture includes at least two convolutional layers.


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def evaluate_accuracy(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            logits = model(x_batch)
            preds = logits.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    return correct / max(total, 1)


model = SimpleCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 3

history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x_batch.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    test_acc = evaluate_accuracy(model, test_loader)
    history.append((epoch, train_loss, test_acc))
    print(f"Epoch {epoch}/{EPOCHS} - loss: {train_loss:.4f} - test_acc: {test_acc:.4f}")

history_df = pd.DataFrame(history, columns=["epoch", "train_loss", "test_acc"])
display(history_df)

In [ ]:
# Visualize first conv layer filters
first_conv_weights = model.features[0].weight.detach().cpu()  # shape: [out_channels, 1, k, k]
num_filters = first_conv_weights.shape[0]

cols = 8
rows = int(np.ceil(num_filters / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.5))
axes = np.array(axes).reshape(-1)

for i in range(len(axes)):
    axes[i].axis("off")
    if i < num_filters:
        kernel = first_conv_weights[i, 0]
        axes[i].imshow(kernel, cmap="gray")
        axes[i].set_title(f"F{i}")

plt.suptitle("First convolution layer filters")
plt.tight_layout()
plt.show()

## 4) Sub-step 4: Readmission risk detector + semantic similarity
This section uses:
- TF-IDF + Logistic Regression classifier for readmission-risk baseline (from patient profile text)
- SentenceTransformer embeddings + cosine similarity for semantic retrieval of clinically similar cases


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

work_df = hospital_df.copy()

# Parse dates robustly and clean mixed-format BMI values
work_df["admission_date_parsed"] = pd.to_datetime(work_df["admission_date"], errors="coerce", dayfirst=True)
work_df["bmi_num"] = (
    work_df["bmi"].astype(str).str.replace("kg/m2", "", regex=False).str.strip()
)
work_df["bmi_num"] = pd.to_numeric(work_df["bmi_num"], errors="coerce")

# Fill missing fields defensively for profile-string creation
for col in ["gender_clean", "department_clean", "insurance_type"]:
    work_df[col] = work_df[col].astype(str).fillna("unknown")

numeric_cols = [
    "age", "length_of_stay_days", "systolic_bp", "diastolic_bp", "glucose_mg_dl",
    "creatinine_mg_dl", "bmi_num", "num_medications", "num_diagnoses", "icu_stay"
]
for col in numeric_cols:
    work_df[col] = pd.to_numeric(work_df[col], errors="coerce")
    work_df[col] = work_df[col].fillna(work_df[col].median())

# Build profile text so TF-IDF/embeddings can operate on structured data
work_df["profile_text"] = (
    "department " + work_df["department_clean"].astype(str)
    + " gender " + work_df["gender_clean"].astype(str)
    + " insurance " + work_df["insurance_type"].astype(str).str.lower().fillna("unknown")
    + " age " + work_df["age"].round(0).astype(int).astype(str)
    + " los " + work_df["length_of_stay_days"].round(0).astype(int).astype(str)
    + " sbp " + work_df["systolic_bp"].round(0).astype(int).astype(str)
    + " dbp " + work_df["diastolic_bp"].round(0).astype(int).astype(str)
    + " glucose " + work_df["glucose_mg_dl"].round(0).astype(int).astype(str)
    + " creatinine " + work_df["creatinine_mg_dl"].round(2).astype(str)
    + " bmi " + work_df["bmi_num"].round(1).astype(str)
    + " meds " + work_df["num_medications"].round(0).astype(int).astype(str)
    + " diagnoses " + work_df["num_diagnoses"].round(0).astype(int).astype(str)
    + " icu " + work_df["icu_stay"].round(0).astype(int).astype(str)
)

work_df["risk_binary"] = pd.to_numeric(work_df[TARGET_COL], errors="coerce").fillna(0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    work_df["profile_text"],
    work_df["risk_binary"],
    test_size=0.2,
    random_state=SEED,
    stratify=work_df["risk_binary"],
)

vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Handle class imbalance via class_weight='balanced'
risk_clf = LogisticRegression(max_iter=500, class_weight="balanced")
risk_clf.fit(X_train_tfidf, y_train)

y_pred = risk_clf.predict(X_test_tfidf)
print(classification_report(y_test, y_pred, digits=4))

# Semantic retrieval using sentence embeddings on patient profile text
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
corpus_texts = work_df["profile_text"].tolist()
corpus_embeddings = embedding_model.encode(corpus_texts, show_progress_bar=False)

risk_examples = work_df[work_df["risk_binary"] == 1]["profile_text"].head(1).tolist()
if risk_examples:
    query = risk_examples[0]
    query_emb = embedding_model.encode([query], show_progress_bar=False)
    scores = cosine_similarity(query_emb, corpus_embeddings)[0]

    top_k = 10
    top_idx = np.argsort(scores)[-top_k:][::-1]
    retrieval_df = pd.DataFrame({
        "profile_text": [corpus_texts[i] for i in top_idx],
        "similarity": [scores[i] for i in top_idx],
        "risk_binary": [work_df.iloc[i]["risk_binary"] for i in top_idx],
    })

    print("High-risk query profile:\n", query)
    display(retrieval_df)
else:
    print("No positive risk examples found for retrieval demo.")

## 5) Sub-step 5: Two-stage clinical risk pipeline
Stage 1: classifier flags likely 30-day readmission risk.  
Stage 2: embedding similarity retrieves clinically similar high-risk cases missed by Stage 1.


In [ ]:
# Stage 1 predictions on full corpus
X_all_tfidf = vectorizer.transform(work_df["profile_text"])
work_df = work_df.reset_index(drop=True)
work_df["stage1_pred"] = risk_clf.predict(X_all_tfidf)

# Stage 2: retrieval for each stage-1 positive seed
SIM_THRESHOLD = 0.70
MAX_SEEDS = 30  # controls runtime of similarity expansion

seed_indices = work_df.index[work_df["stage1_pred"] == 1].tolist()[:MAX_SEEDS]
stage2_added = set()

for seed_idx in seed_indices:
    q_emb = corpus_embeddings[seed_idx : seed_idx + 1]
    sims = cosine_similarity(q_emb, corpus_embeddings)[0]
    similar_idx = np.where(sims >= SIM_THRESHOLD)[0]
    for idx in similar_idx:
        if work_df.loc[idx, "stage1_pred"] == 0:
            stage2_added.add(int(idx))

work_df["stage2_added"] = 0
if stage2_added:
    work_df.loc[list(stage2_added), "stage2_added"] = 1

work_df["final_flag"] = ((work_df["stage1_pred"] == 1) | (work_df["stage2_added"] == 1)).astype(int)

# Evaluate how many true readmissions Stage 2 recovers
true_risk = work_df["risk_binary"] == 1
stage1_hits = ((work_df["stage1_pred"] == 1) & true_risk).sum()
final_hits = ((work_df["final_flag"] == 1) & true_risk).sum()
additional_hits = final_hits - stage1_hits

precision, recall, f1, _ = precision_recall_fscore_support(
    work_df["risk_binary"], work_df["final_flag"], average="binary", zero_division=0
)

print("Stage 1 true readmissions captured:", int(stage1_hits))
print("Final pipeline true readmissions captured:", int(final_hits))
print("Additional true readmissions surfaced by Stage 2:", int(additional_hits))
print(f"Pipeline precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")

# Daily estimate for 100,000 cases/day
flag_rate = work_df["final_flag"].mean()
estimated_daily_reviews = int(round(100_000 * flag_rate))
print("Estimated daily review volume at 100,000 cases/day:", estimated_daily_reviews)

display(
    work_df[
        [
            "patient_id", "profile_text", "risk_binary",
            "stage1_pred", "stage2_added", "final_flag"
        ]
    ].head(20)
)

## Recommendation (fill after running)
- Most appropriate metric for readmission-risk operations goal:
- Why this metric matters more than alternatives:
- Additional true readmissions surfaced by Stage 2:
- Estimated daily review workload at 100k cases/day:
- Final recommendation:
